# Mix Roboflow Universe datasets with a fixed per-batch ratio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roboflow/rf-detr/blob/develop/docs/cookbooks/multi-source-batch-sampler.ipynb)

Training on several COCO datasets through a plain `ConcatDataset` draws each source in proportion
to its size. A large public export then dominates every batch, and a small hand-labelled set barely
affects the gradient.

`WeightedMultiSourceBatchSampler` keeps the **per-source composition of every batch** fixed instead.
This cookbook downloads three detection datasets from [Roboflow Universe](https://universe.roboflow.com),
unifies their class names, and fine-tunes `RFDETRSmall` so that every batch is `4 / 3 / 1` samples from
the three sources — regardless of how their sizes compare.

See [Training customization → Mixing datasets](../learn/train/customization.md) for the API reference.

## Setup

Install `rfdetr` with the `train` extra (PyTorch Lightning, COCO eval) plus `roboflow` for the
Universe download. A GPU is recommended; drop `BATCH_SIZE` if you hit out-of-memory.

In [ ]:
!pip install -q "rfdetr[train]>=1.10.0" roboflow

In [ ]:
"""Mix three Roboflow Universe datasets with WeightedMultiSourceBatchSampler."""

import json
import os
from collections import Counter
from pathlib import Path
from typing import Any

from pytorch_lightning import Callback, Trainer
from torch.utils.data import ConcatDataset

from rfdetr import RFDETRSmall
from rfdetr.config import TrainConfig
from rfdetr.datasets import WeightedMultiSourceBatchSampler, compute_source_batch_sizes
from rfdetr.datasets.coco import (
    CocoDetection,
    annotated_category_ids,
    filter_parent_categories,
    make_coco_transforms,
)
from rfdetr.datasets.kornia_transforms import is_gpu_postprocess, resolve_backend_for_build
from rfdetr.training import RFDETRDataModule, RFDETRModelModule, build_trainer
from rfdetr.utilities.reproducibility import seed_all

In [ ]:
PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
DATASETS_DIR = PROJECT_ROOT / "datasets"
OUTPUT_DIR = PROJECT_ROOT / "output" / "multi_source_batch_sampler"

# Three Universe COCO detection exports already used in the fine-tune cookbook, so the
# download paths are known to work. They differ in size and class set — that is the point.
SOURCES: list[dict[str, Any]] = [
    {
        "key": "hard_hat_ppes",
        "name": "Hard Hat Worker Safety",
        "workspace": "safety-first",
        "project": "hard-hat-worker-safety-equipments",
        "version": 9,
        "source_url": "https://universe.roboflow.com/safety-first/hard-hat-worker-safety-equipments",
        "weight": 0.5,
    },
    {
        "key": "traffic",
        "name": "Traffic Detection",
        "workspace": "redlightrunningdection",
        "project": "traffic-detection-sutq6",
        "version": 36,
        "source_url": "https://universe.roboflow.com/redlightrunningdection/traffic-detection-sutq6",
        "weight": 0.375,
    },
    {
        "key": "football",
        "name": "Football Player Detection",
        "workspace": "football-gozni",
        "project": "football-player-detection-bfswn",
        "version": 1,
        "source_url": "https://universe.roboflow.com/football-gozni/football-player-detection-bfswn",
        "weight": 0.125,
    },
]

SEED = 7
EPOCHS = 5
BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 2
NUM_WORKERS = 2
RESOLUTION = 512
LR = 1e-4
LR_ENCODER = 1e-4
WEIGHTS = [float(source["weight"]) for source in SOURCES]

print(f"sources={[source['key'] for source in SOURCES]}")
print(f"weights={WEIGHTS}")
print(f"slots_per_batch={compute_source_batch_sizes(BATCH_SIZE, WEIGHTS)}")

## 1 - Download the datasets

Get a free Roboflow API key at `app.roboflow.com/settings/api` and expose it as `ROBOFLOW_API_KEY`
(an environment variable locally, or a Colab secret with the same name). Each download is idempotent —
re-running skips the transfer if the target directory already exists.

In [ ]:
try:
    from google.colab import userdata

    try:
        ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY") or ""
    except Exception:
        ROBOFLOW_API_KEY = ""
except ImportError:
    ROBOFLOW_API_KEY = ""

if not ROBOFLOW_API_KEY:
    ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY", "")
if not ROBOFLOW_API_KEY:
    raise RuntimeError(
        "ROBOFLOW_API_KEY not found. "
        "In Colab: add it via Secrets (key icon). "
        "Locally: set the environment variable before running."
    )

from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
for source in SOURCES:
    location = DATASETS_DIR / str(source["key"])
    dataset = (
        rf.workspace(str(source["workspace"]))
        .project(str(source["project"]))
        .version(int(source["version"]))
        .download("coco", location=str(location))
    )
    source["dataset_dir"] = Path(dataset.location)
    source["train_ann"] = source["dataset_dir"] / "train" / "_annotations.coco.json"
    print(f"{source['key']}: {source['dataset_dir']}")

## 2 - Unify class names

Each Universe export remaps categories to its own `0..N-1` independently, and Roboflow prepends a
synthetic grouping category that RF-DETR drops. Concatenating the datasets as-is would mix those
label spaces. Build one class list (first-seen name wins) and a per-source `category_id → label`
map so every source writes into the same head.

In [ ]:
def _kept_categories(ann_path: Path) -> list[dict[str, Any]]:
    with ann_path.open() as handle:
        coco = json.load(handle)
    return filter_parent_categories(coco["categories"], annotated_category_ids(coco))


CLASS_NAMES: list[str] = []
name_to_index: dict[str, int] = {}
for source in SOURCES:
    source["cat2label"] = {}
    for category in _kept_categories(Path(source["train_ann"])):
        name = str(category["name"])
        key = name.casefold()
        if key not in name_to_index:
            name_to_index[key] = len(CLASS_NAMES)
            CLASS_NAMES.append(name)
        source["cat2label"][int(category["id"])] = name_to_index[key]

NUM_CLASSES = len(CLASS_NAMES)
print(f"num_classes={NUM_CLASSES}")
print(f"class_names={CLASS_NAMES}")
for source in SOURCES:
    names = [CLASS_NAMES[index] for index in sorted(set(source["cat2label"].values()))]
    print(f"{source['key']}: {names}")

## 3 - Subclass `RFDETRDataModule`

`setup` concatenates one `CocoDetection` per source. The `build_train_sampler` hook then returns a
`WeightedMultiSourceBatchSampler` instead of a size-proportional shuffle, and RF-DETR's own
`train_dataloader` builds the DataLoader around it - so the loader keeps its configured
`collate_fn`, worker count, pinning and per-worker augmentation seeding without restating any of them.
`epoch_length="smallest"` keeps this demo short: the epoch ends when the smallest source has been
seen once, so the larger sources are sub-sampled rather than the small one being recycled many times.

> **Note:** Under DDP pass `use_distributed_sampler=False` to `build_trainer`. The sampler shards
> batches itself via `num_replicas` / `rank`; a second `DistributedSampler` would split the data twice.
>
> Lightning does not call `set_epoch` on a batch sampler you construct yourself. The tiny callback
> below wires that up so the within-source shuffle changes every epoch.

In [ ]:
_SPLIT_DIRS = {
    "train": "train",
    "val": "valid",
    "test": "test",
}


def _build_source(
    source: dict[str, Any],
    image_set: str,
    *,
    resolution: int,
    model_config: Any,
    train_config: TrainConfig,
) -> CocoDetection:
    root = Path(source["dataset_dir"])
    split_dir = root / _SPLIT_DIRS[image_set]
    gpu_postprocess = is_gpu_postprocess(resolve_backend_for_build(train_config.augmentation_backend))
    return CocoDetection(
        split_dir,
        split_dir / "_annotations.coco.json",
        transforms=make_coco_transforms(
            image_set,
            resolution,
            multi_scale=train_config.multi_scale,
            expanded_scales=train_config.expanded_scales,
            skip_random_resize=not train_config.do_random_resize_via_padding,
            patch_size=model_config.patch_size,
            num_windows=model_config.num_windows,
            aug_config=train_config.aug_config,
            scale_jitter=train_config.scale_jitter,
            gpu_postprocess=gpu_postprocess,
        ),
        remap_category_ids=True,
        cat2label=source["cat2label"],
    )


class MultiSourceDataModule(RFDETRDataModule):
    """Train on several COCO sources with a fixed per-batch ratio."""

    def __init__(
        self,
        model_config: Any,
        train_config: TrainConfig,
        sources: list[dict[str, Any]],
        weights: list[float],
    ) -> None:
        super().__init__(model_config, train_config)
        self.sources = sources
        self.weights = weights
        self.batch_sampler: WeightedMultiSourceBatchSampler | None = None

    def setup(self, stage: str) -> None:
        super().setup(stage)
        if stage != "fit":
            return
        train_sources = [
            _build_source(
                source,
                "train",
                resolution=self.model_config.resolution,
                model_config=self.model_config,
                train_config=self.train_config,
            )
            for source in self.sources
        ]
        self._dataset_train = ConcatDataset(train_sources)
        self._source_sizes = [len(dataset) for dataset in train_sources]
        print(
            "source_sizes="
            + ", ".join(
                f"{source['key']}={size}" for source, size in zip(self.sources, self._source_sizes, strict=True)
            )
        )

    def build_train_sampler(self, dataset: Any) -> WeightedMultiSourceBatchSampler:
        """Return the batch sampler; the base ``train_dataloader`` builds the DataLoader around it.

        Using the hook rather than overriding ``train_dataloader`` keeps every loader setting RF-DETR
        already configures - including ``worker_init_fn``, which reseeds NumPy and ``random`` per worker
        so augmentation does not repeat across workers.
        """
        world_size = self.trainer.world_size if self.trainer else 1
        rank = self.trainer.global_rank if self.trainer else 0
        self.batch_sampler = WeightedMultiSourceBatchSampler.from_concat_dataset(
            dataset,
            weights=self.weights,
            batch_size=self._resolve_batch_size(),
            num_replicas=world_size,
            rank=rank,
            seed=SEED,
            epoch_length="smallest",
            # Keep every gradient-accumulation window complete: the hook skips the dataset padding the
            # default path relies on, so the sampler has to round its own epoch to a whole number of windows.
            batch_multiple=self.train_config.grad_accum_steps,
        )
        return self.batch_sampler


class SetBatchSamplerEpoch(Callback):
    """Call ``set_epoch`` on the multi-source batch sampler at the start of every epoch."""

    def __init__(self, datamodule: MultiSourceDataModule) -> None:
        super().__init__()
        self._datamodule = datamodule

    def on_train_epoch_start(self, trainer: Trainer, _pl_module: Any) -> None:
        sampler = self._datamodule.batch_sampler
        if sampler is not None:
            sampler.set_epoch(trainer.current_epoch)

## 4 - Inspect one epoch of batches

Before training, iterate the sampler and count how many indices came from each source.
Every batch should match `compute_source_batch_sizes`. A size-proportional `ConcatDataset`
shuffle would instead follow the source-size ratios printed below.

In [ ]:
seed_all(SEED)
variant = RFDETRSmall(  # type: ignore[no-untyped-call]
    num_classes=NUM_CLASSES,
    resolution=RESOLUTION,
)
variant.model_config.model_name = type(variant).__name__

PRIMARY_DIR = Path(SOURCES[0]["dataset_dir"])
train_config = TrainConfig(
    dataset_file="roboflow",
    dataset_dir=str(PRIMARY_DIR),
    output_dir=str(OUTPUT_DIR),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    num_workers=NUM_WORKERS,
    lr=LR,
    lr_encoder=LR_ENCODER,
    warmup_epochs=1,
    use_ema=False,
    run_test=False,
    multi_scale=False,
    expanded_scales=False,
    tensorboard=False,
    wandb=False,
    mlflow=False,
    clearml=False,
    class_names=CLASS_NAMES,
    progress_bar="tqdm",
)

datamodule = MultiSourceDataModule(variant.model_config, train_config, SOURCES, WEIGHTS)
datamodule.setup("fit")
datamodule.train_dataloader()
sampler = datamodule.batch_sampler
assert sampler is not None

expected_slots = tuple(compute_source_batch_sizes(BATCH_SIZE, WEIGHTS))
source_sizes = sampler.source_sizes
total_images = sum(source_sizes)
print(f"expected_slots_per_batch={list(expected_slots)}")
print(f"size_proportional_share={[round(size / total_images, 3) for size in source_sizes]}")
print(f"batches_this_epoch={len(sampler)}")

header = f"{'batch':>5}  " + "  ".join(f"{str(source['key']):>16}" for source in SOURCES)
print(header)
compositions: list[tuple[int, ...]] = []
drawn: Counter[int] = Counter()
for batch_index, batch in enumerate(sampler):
    counts = [0] * len(source_sizes)
    for index in batch:
        start = 0
        for source_index, size in enumerate(source_sizes):
            if start <= index < start + size:
                counts[source_index] += 1
                drawn[source_index] += 1
                break
            start += size
    composition = tuple(counts)
    compositions.append(composition)
    if batch_index < 8:
        print(f"{batch_index:>5}  " + "  ".join(f"{count:>16}" for count in composition))
if len(compositions) > 8:
    print(f"... ({len(compositions) - 8} more batches)")

unique = set(compositions)
print(f"unique_batch_compositions={unique}")
print("samples_drawn_this_epoch=" + str({str(SOURCES[index]["key"]): drawn[index] for index in range(len(SOURCES))}))
assert unique == {expected_slots}, unique

## 5 - Fine-tune

Validation is skipped here (`limit_val_batches=0`): the val split of the first source does not cover
classes that only appear in the other two, so COCO eval on that split would be misleading. Raise
`EPOCHS` and restore validation once you have a held-out mix you actually want to score.

In [ ]:
model = RFDETRModelModule(variant.model_config, train_config)
trainer = build_trainer(
    train_config,
    variant.model_config,
    use_distributed_sampler=False,
    num_sanity_val_steps=0,
    limit_val_batches=0,
)
trainer.callbacks.append(SetBatchSamplerEpoch(datamodule))
trainer.fit(model, datamodule=datamodule)

## What to try next

- Change `WEIGHTS` (they need not sum to 1) and re-run the inspection cell — `source_batch_sizes`
  always sums to `BATCH_SIZE`.
- Pass `epoch_length="largest"` to see every image of the biggest source once per epoch; the
  smallest source will then be recycled, and the sampler logs a warning past a 10× recycle factor.
- Swap the three Universe projects for your own COCO exports; keep the name-union step so labels
  stay aligned.
- Full API: [`WeightedMultiSourceBatchSampler`](../reference/training.md) and
  [Mixing datasets with a fixed per-batch ratio](../learn/train/customization.md).